# CTAI 直肠癌 CT 肿瘤分割 — Kaggle GPU 训练

## 使用前准备
1. 将 `直肠癌数据/` 文件夹上传为 Kaggle Dataset（命名为 `rectal-caner-data`）
2. 将 `CTAI_model/` 代码文件夹压缩为 zip 上传为另一个 Kaggle Dataset（命名为 `ctai-model-code`）
3. 新建 Notebook → Settings → Accelerator → GPU T4 x2
4. Add Data → 添加上面两个 Dataset
5. **按顺序逐个运行 Cell，不要直接 Run All**

> ⚠️ Cell 1 安装依赖后 Kaggle 可能重启 kernel，重启后从 Cell 2 继续运行即可。

In [ ]:
# Cell 1: 安装依赖（单独运行，可能触发 kernel 重启，重启后从 Cell 2 继续）
import subprocess, sys
pkgs = ['albumentations', 'SimpleITK', 'tqdm', 'pydicom', 'scikit-image', 'scipy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('依赖安装完成')

In [ ]:
# Cell 2: 环境检查
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 3: 复制代码到工作目录（Kaggle input 是只读的）
import os, sys, shutil, glob

# 找到代码目录
code_candidates = [
    '/kaggle/input/ctai-model-code/CTAI_model',
    '/kaggle/input/ctai-model-code',
]
code_src = None
for c in code_candidates:
    if os.path.isfile(os.path.join(c, 'train.py')):
        code_src = c
        break

if code_src is None:
    found = glob.glob('/kaggle/input/**/train.py', recursive=True)
    if found:
        code_src = os.path.dirname(found[0])
    else:
        raise FileNotFoundError('找不到代码目录，请检查 Dataset 名称是否为 ctai-model-code')

work_dir = '/kaggle/working/CTAI_model'
if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
shutil.copytree(code_src, work_dir)
os.chdir(work_dir)
sys.path.insert(0, work_dir)  # 确保 import 能找到本地模块
print(f'代码目录: {work_dir}')
print(f'文件: {os.listdir(".")}')

In [ ]:
# Cell 4: 检查数据目录
data_candidates = [
    '/kaggle/input/rectal-caner-data/rectal_data',
    '/kaggle/input/rectal-caner-data/直肠癌数据',
    '/kaggle/input/rectal-caner-data',
    '/kaggle/input/rectal-cancer-data/rectal_data',
]
data_dir = None
for d in data_candidates:
    if os.path.isdir(d):
        subs = os.listdir(d)
        if any(s.isdigit() for s in subs):
            data_dir = d
            break

if data_dir is None:
    dcm = glob.glob('/kaggle/input/**/*.dcm', recursive=True)
    if dcm:
        data_dir = os.path.dirname(os.path.dirname(os.path.dirname(dcm[0])))
    else:
        raise FileNotFoundError('找不到数据目录，请检查 Dataset 名称是否为 rectal-caner-data')

patients = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
dcm_files = glob.glob(os.path.join(data_dir, '**/*.dcm'), recursive=True)
mask_files = glob.glob(os.path.join(data_dir, '**/*_mask.png'), recursive=True)
print(f'数据目录: {data_dir}')
print(f'患者数: {len(patients)}')
print(f'DICOM 文件: {len(dcm_files)}')
print(f'Mask 文件: {len(mask_files)}')

In [ ]:
# Cell 5: 开始训练（Kaggle T4 x2 优化参数）
#
# 参数说明:
#   --batch_size 8         T4 16GB 显存可跑 8
#   --repeats 5            全量数据不需要多次重复
#   --epochs 100           全量数据 100 epoch 足够
#   --eval_interval 5      每 5 epoch 快速评估
#   --full_eval_interval 25 每 25 epoch 全量评估
#   --train_mode mixed     混合模式（肿瘤切片 + 部分背景切片）
#   --num_workers 2        Kaggle 可用 2 个 worker
#   --val_max_slices 80    快速评估最多 80 张含肿瘤切片

!python /kaggle/working/CTAI_model/train.py \
    --batch_size 8 \
    --repeats 5 \
    --epochs 100 \
    --eval_interval 5 \
    --full_eval_interval 25 \
    --train_mode mixed \
    --mixed_ratio 0.3 \
    --lr 3e-4 \
    --deep_supervision True \
    --use_ema True \
    --num_workers 2 \
    --val_max_slices 80 \
    --patience 10

In [ ]:
# Cell 6: 查看训练曲线
from IPython.display import Image, display
import os

curve_path = '/kaggle/working/logs/training_curves.png'
if os.path.exists(curve_path):
    display(Image(filename=curve_path))
else:
    print('训练曲线尚未生成，训练完成后再运行此 cell')

In [ ]:
# Cell 7: 训练完成后 — 全量 TTA 评估
!python /kaggle/working/CTAI_model/evaluate.py \
    --weights /kaggle/working/checkpoints/best_model.pth \
    --tta \
    --output_dir /kaggle/working/evaluation

In [ ]:
# Cell 8: 查看评估结果
import json

report_path = '/kaggle/working/evaluation/evaluation_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    print('=== 评估摘要 ===')
    for k, v in report['summary'].items():
        print(f'  {k}: {v}')
else:
    print('评估报告不存在，请先运行 Cell 7')

In [ ]:
# Cell 9: 打包模型，从 Output 标签页下载
import shutil

shutil.make_archive('/kaggle/working/ctai_results', 'zip', '/kaggle/working/checkpoints')
print('模型已打包: /kaggle/working/ctai_results.zip')
print('请在右侧 Output 标签页点击下载')